In [ ]:
from dotenv import load_dotenv
load_dotenv() # reads the key-value pairs from your .env file and loads them into the current process's environment variables. It returns True if it successfully found and loaded a .env file, or False if it couldn't find one

True

In [122]:
from openai import OpenAI # imports the OpenAI client class from the openai Python package.
openai_client = OpenAI() # creates an instance of that client, which handles authentication and communication with OpenAI's API.

# Plain LLMs lack our data

In [123]:
# First, let's define a function to talk to the LLM. This LLM fucntion lets us to interact with the LLM provider

In [124]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [125]:
llm("Hey, what's up?")

'Hey! Not much — I’m here and ready to help. What’s on your mind?'

In [126]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—if the course is still open, you can usually join now.

A couple quick notes:
- **Check the enrollment deadline** or whether late enrollment is allowed.
- If it’s already in progress, you may need to **catch up on missed material**.
- If this is a **class or program with limited spots**, availability may depend on capacity.

If you want, I can help you draft a quick message to the instructor or admissions team asking to join.


In [127]:
#These are the things we think will be helpful for the LLM to answer the question

context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [128]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [129]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [130]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


In [131]:
# RAG Architecture 

# That's the entire architecture. It comes down to three components.

# The pieces are search, the prompt, and the LLM:

# search
# prompt
# LLM

In [132]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [133]:
# The FAQ data is available as JSON from the DataTalks.Club website. The FAQ data has been made available as a JSON endpoint 
# Now lets fetch the data

In [134]:
# This returns a list of courses. Each course has a path field that points to its FAQ data. This will fetch all the FAQ documents for all courses.

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [135]:
courses_raw # This returns a list of courses. Each course has a path field that points to its FAQ data.

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 113},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 253},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 471}]

In [136]:
# This code loops through a list of courses and downloads/collects FAQ data for each one from a set of URLs. 
documents = [] # Creates an empty list that will hold all the collected FAQ documents from every course.
url_prefix = "https://datatalks.club/faq" # The base URL that all course-specific URLs will be built from.

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status() #raise an issue if something isn't working. Dont continue.
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1375

In [137]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [138]:
# We do not want to send all documents to LLM cos would be very expensive and not effective 
# We will index the data so only the most relevant docs are retrieved 
# Searching matters because where you have loads of documents. Sending all of them to the LLM would be expensive and slow. 
# The model would get confused with too much data. Search finds the most relevant documents to send instead.

In [139]:
# There are many search libraries you can use - Apache Lucene, Elasticsearch, Solr, and others. 
# But these are somewhat heavy. For example, to run Elasticsearch, you need to start a Docker container.
# Elastic search is based on Lucene. So it uses Lucene under the hood
# There is a light weight alternatice called minsearch. minsearch is a simple in-memory search engine. It's lightweight, 
# so it runs anywhere Python runs, including Google Colab where you can't start a Docker container. 
# It's a toy implementation, not production ready, but it illustrates how search engines work and it gives good results.

In [140]:
# Text fields are the fields you search through. When you type a query, the search engine looks at these fields and tokenizes them. 
# It splits text into words, lowercases them, removes stop words, and ranks the results by relevance. So question, section, and answer 
# are text fields.

# Keyword fields are for exact matching. Think of a SQL query like SELECT * FROM index WHERE course = 'data-engineering-zoomcamp'. 
# The results must come from the specified course, no matter what ranking or boosting you do for text fields.


from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [141]:
# index.search(question)

In [142]:
index.search (question, filter_dict={'course': 'llm-zoomcamp'},
              num_results = 5)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [143]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [144]:
# We used boost_dict to give the question field more weight (2.0 instead of the default 1.0) and section less weight (0.5). 
# Query words appearing in the question field is a stronger signal than them appearing in the section name.
# We used filter_dict to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.

In [145]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [146]:
search_results = search(question)

In [147]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [148]:
# Building the context
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [149]:
# The user prompt template

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [150]:
# Building the prompt

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()


In [151]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [152]:

# Now we have prompt, We send it to the LLM:

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

# Exploring the response

In [153]:
# The response is a Pydantic object. The answer is in response.output - a list of output items.

In [154]:
response.output_text 

'Yes — you can join now.\n\nYou can start learning anytime, and if the submission form is still open, you can still submit homework/project work. If you want a certificate, make sure you submit your project while submissions are still being accepted and join with a live cohort (certificates aren’t available for self-paced participation).'

In [155]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0711dc21acf3dee0006a5281eb216481a2a301aea06a0e4207",
  "created_at": 1783792107.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0711dc21acf3dee0006a5281eba1b081a2abaf2575588c49ff",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can join now.\n\nYou can start learning anytime, and if the submission form is still open, you can still submit homework/project work. If you want a certificate, make sure you submit your project while submissions are still being accepted and join with a live cohort (certificates aren’t available for self-paced participation).",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperatu

In [156]:
response.usage # The usage counts tell you how many tokens the request consumed:

ResponseUsage(input_tokens=678, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=69, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=747)

In [157]:

# calculate the cost of the request we just made: Below is the price for gpt-5.4-mini:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0008190000000000001

In [158]:
# This particular request costs a fraction of a cent. Even a full RAG query with a long prompt stays under $0.01. We need to send a lot of queries to even spend one cent. These models are cheap to play with.

# The LLM function

In [159]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [160]:
response.output_text

'Yes — you can join now.\n\nYou can start learning anytime, and if the submission form is still open, you can still submit homework/project work. If you want a certificate, make sure you submit your project while submissions are still being accepted and join with a live cohort (certificates aren’t available for self-paced participation).'

# Full RAG

In [161]:
# With search, the prompt, and the LLM ready, we wire them together:

def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer